# 08: Diagnóstico de Subgrupos, Validação Externa (OpenML 43439) e Teste de Recuperação Estrutural

Este notebook complementa os experimentos principais do TCC (`04_experimentos.ipynb` e resultados sem sobreposição em `resultados/artigo_sem_sobreposicao/completo/`) com três análises metodológicas rigorosas:

1. **Diagnóstico por Subgrupos Cadastrais na Base Primária (Teste Temporal 2021–2022, $n = 9.950$):**
   - Avalia o efeito do apagão cadastral de 2021–2022 ($4.104$ agendamentos sem `patient_id`, equivalentes a $41{,}2\%$ do teste temporal) utilizando o cálculo exato de orçamento por esperança matemática em empates (`classification_report_budget`), sem artefatos de ordenação (`argsort`) e reportando intervalos de confiança via *bootstrap* e o ganho normalizado pela prevalência (*Lift de AP* = $\text{AP} / \text{Prevalência}$).
2. **Validação Externa em Base Pública com *Lead Time* (`OpenML id=43439` / Hoppen, 2016, $n = 110.526$):**
   - Avalia o mesmo classificador `DiscreteBN` com seleção estrutural pelo Manto de Markov em uma base pública real que registra a data de criação do agendamento (`ScheduledDay`) e a data da consulta (`AppointmentDay`), permitindo extrair `dias_antecedencia` (*lead time*), além do indicador de envio de lembrete (`SMS_received`).
   - **Controle de confundimento e linha de base:** A cópia distribuída no OpenML (`id=43439`) não possui identificador de paciente (`PatientId`), não permitindo construir histórico longitudinal, e `SMS_received` indica lembrete enviado (associado a maior taxa de ausência no agregado, $27{,}6\%$ contra $16{,}7\%$, porque o SMS não era enviado a agendamentos com até 2 dias de antecedência; dentro da mesma faixa de espera, quem recebeu SMS faltou menos), e não confirmação ativa. Por isso, comparamos a Rede Bayesiana diretamente contra **(a) `dias_antecedencia` isolado** e **(b) Regressão Logística** nas mesmas 5 dobras de `StratifiedKFold`, separando o efeito da base de dados do efeito do modelo.
3. **Teste de Recuperação Estrutural sob Gerador Causal Conhecido (Cenário 1B: $X \to Y_{\text{sim}}$ contra Oráculo Teórico):**
   - Para verificar se o aprendizado estrutural (*Hill-Climb Search* + Manto de Markov) recupera corretamente preditores operacionais quando o processo gerador é conhecido (**sem incorrer em condicionamento reverso $Y \to X$, que geraria vazamento de alvo circular**), simulamos o desfecho $Y_{\text{sim}}$ diretamente a partir de $X$ ($X \to Y_{\text{sim}}$) e comparamos a Rede Bayesiana contra o **limite máximo teórico do gerador (Oráculo de Bayes com a probabilidade verdadeira $p_{\text{true}}$)**.

---

## Referências das Bases de Dados Utilizadas

1. **Base Primária (Clínica de Reabilitação, Sul do Brasil):**
   - **Fonte:** Salazar, L. H. A.; Fernandes, A. M. R.; Dazzi, R. L. S.; Garcia, N. M.; Leithardt, V. R. Q. (2023). *Medical Appointments No-Show*. **Mendeley Data**, V1. DOI: `10.17632/73c6373HF2.1`. (Licença CC BY 4.0; $n = 49.593$ agendamentos entre 2016 e 2022; prevalência global de $9{,}74\%$ de *no-show*).
   - **Artigo associado:** Salazar et al. (2022). *Using Different Models of Machine Learning to Predict Attendance at Medical Appointments*. **Journal of Information Systems Engineering and Management**, 7(4), 18392.
2. **Base Pública Externa (Agendamentos Ambulatoriais, Vitória/ES, 2016):**
   - **Fonte original:** Hoppen, J. (2016). *Medical Appointment No Shows*. **Kaggle Dataset** (`joniarroba/noshowappointments`, Aquarela Advanced Analytics; agendamentos realizados entre 29/04/2016 e 08/06/2016).
   - **Distribuição utilizada:** espelho público no **OpenML** (`data_id=43439`, submetido em 2022 sob licença CC BY-NC-SA 4.0, sem as colunas `PatientId` e `AppointmentID`, com $110.526$ registros válidos após exclusão de $5$ registros com antecedência negativa e idade inválida; prevalência de $20{,}19\%$ de *no-show*).
3. **Literatura sobre Antecedência de Marcação (*Lead Time*):**
   - Dantas, L. F.; Fleck, J. L.; Cyrino Oliveira, F. L.; Hamacher, S. (2018). *No-shows in appointment scheduling: a systematic literature review*. **Health Policy**, 122(4), 412–421.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import fetch_openml

from noshow.data import load_raw, add_patient_proxy, TARGET
from noshow.weather import fix_weather
from noshow.features import (
    CONTINUOUS, CATEGORICAL, Discretizer,
    add_age, add_calendar, group_categories, add_service_time, add_patient_history
)
from noshow.preprocessing import FoldCategories
from noshow.bayes import DiscreteBN, learn_structure, markov_blanket
from noshow.pipeline import classification_report_budget
from noshow.experiments import run_config

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()

def load_clinic_basic() -> pd.DataFrame:
    df = add_patient_proxy(fix_weather(load_raw()))
    for fn in (add_age, add_calendar, group_categories, add_service_time, add_patient_history):
        df = fn(df)
    return df

def evaluate_predictions(name: str, y_true: np.ndarray, p_pred: np.ndarray, manto_str: str, n_params: float) -> dict:
    """Calcula métricas de discriminação, calibração, Lift de AP e orçamento com desempate exato."""
    y_true = np.asarray(y_true, dtype=int)
    p_pred = np.asarray(p_pred, dtype=float)
    prev = float(y_true.mean())
    roc = float(roc_auc_score(y_true, p_pred)) if len(np.unique(p_pred)) > 1 else 0.5
    ap = float(average_precision_score(y_true, p_pred))
    lift_ap = ap / prev if prev > 0 else 1.0
    brier = float(brier_score_loss(y_true, p_pred))
    b5 = classification_report_budget(y_true, p_pred, fraction=0.05)
    b10 = classification_report_budget(y_true, p_pred, fraction=0.10)
    return {
        'Cenário / Modelo': name,
        'n': int(len(y_true)),
        'Prevalência': prev,
        'roc_auc': roc,
        'ap': ap,
        'lift_ap': lift_ap,
        'brier': brier,
        'prec_top5': float(b5['precision']),
        'rec_top5': float(b5['recall']),
        'prec_top10': float(b10['precision']),
        'rec_top10': float(b10['recall']),
        'Manto': manto_str,
        'Params': float(n_params),
    }

def bootstrap_ci(y_true: np.ndarray, p_pred: np.ndarray, n_boot: int = 1000, seed: int = 42):
    rng = np.random.default_rng(seed)
    y_true, p_pred = np.asarray(y_true, dtype=int), np.asarray(p_pred, dtype=float)
    rocs, aps = [], []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y_true[idx], p_pred[idx]
        if len(np.unique(yb)) < 2:
            continue
        rocs.append(roc_auc_score(yb, pb) if len(np.unique(pb)) > 1 else 0.5)
        aps.append(average_precision_score(yb, pb))
    return (np.percentile(rocs, [2.5, 97.5]), np.percentile(aps, [2.5, 97.5]))

print('Ambiente carregado com sucesso. Diretório raiz:', ROOT)

---
## Parte 1: Diagnóstico por Subgrupos Cadastrais na Base Primária (2021–2022)

Na base primária da clínica (`n = 49.593`), o período de teste temporal (2021–2022, `n = 9.950`, prevalência de $15{,}31\%$) sofre um apagão cadastral: **$4.104$ agendamentos ($41{,}2\%$) não possuem campos demográficos suficientes para reconstruir o `patient_id`**, deixando todas as variáveis históricas (`taxa_faltas_anteriores`, `n_agendamentos_anteriores`, `dias_desde_ultima_consulta`) com estado `ausente`.

Para manter total consistência com a Tabela 1 do artigo, avaliamos os subgrupos utilizando exatamente o mesmo modelo **`C1 Engenharia sem SMOTENC` (BIC, seleção interna: 3 faixas, `ESS = 100`, 7 parâmetros)** da tabela principal:
- **Pacientes SEM `patient_id` ($n = 4.104$):** todas as predições recebem a mesma probabilidade constante da categoria `ausente`, resultando em `ROC-AUC = 0,5000`, `Lift de AP = 1,00x` e `P@10% = Prevalência = 16,5%` sob desempate exato pela esperança matemática.
- **Todos os pacientes COM `patient_id` ($n = 5.846$):** onde há histórico observável.
- **Recorte exploratório COM `patient_id` e 1–9 consultas anteriores ($n = 676$):** recorte pós-hoc de menor tamanho amostral (reportado com intervalo de confiança *bootstrap* de 95%).

In [ ]:
df_clinic = load_clinic_basic()

TEMPORAL_EXCLUDE = {'appointment_year', 'periodo'}
cat_temp = [c for c in CATEGORICAL if c not in TEMPORAL_EXCLUDE]
cont_temp = [c for c in CONTINUOUS if c not in TEMPORAL_EXCLUDE]

mask_tr = (df_clinic['appointment_date'].dt.year <= 2020).to_numpy()
mask_te = (df_clinic['appointment_date'].dt.year >= 2021).to_numpy()

X_eng = df_clinic[cat_temp + cont_temp + ['city']]
Xtr_c = X_eng.loc[mask_tr].reset_index(drop=True)
ytr_c = df_clinic.loc[mask_tr, TARGET].to_numpy(dtype=int)
Xte_c = X_eng.loc[mask_te].reset_index(drop=True)
yte_c = df_clinic.loc[mask_te, TARGET].to_numpy(dtype=int)
te_meta = df_clinic.loc[mask_te, ['patient_id', 'n_agendamentos_anteriores', TARGET]].reset_index(drop=True)

# Divisão interna temporal idêntica a scripts/run_h3.py
dates_tr = df_clinic.loc[mask_tr, 'appointment_date'].reset_index(drop=True)
cut_date = dates_tr.sort_values().iloc[int(0.75 * len(dates_tr))]
inner_split = (np.where(dates_tr < cut_date)[0], np.where(dates_tr >= cut_date)[0])

# 1) C3 SMOTENC sem escalonador (BIC) — líder temporal do protocolo sem sobreposição
res_c3_sem = run_config(
    'C3 engenharia + SMOTENC (sem escalonador)', Xtr_c, ytr_c, Xte_c, yte_c,
    cat_temp, cont_temp, scaler='sem', smote=True, scoring='bic-d', seed=42,
    repair_derived=True, correct_prior=True, coherent_history=True,
    inner_split=inner_split, ess_grid=(1.0, 10.0, 100.0), n_bins_grid=(3, 5), sampling_grid=(0.25, 0.5, 1.0)
)

# 2) C1 Engenharia sem SMOTENC (BIC) com seleção interna (seleciona n_bins=3, ess=100, 7 parâmetros)
res_c1_grid = run_config(
    'C1 engenharia (seleção interna)', Xtr_c, ytr_c, Xte_c, yte_c,
    cat_temp, cont_temp, scaler='sem', smote=False, scoring='bic-d', seed=42,
    inner_split=inner_split, ess_grid=(1.0, 10.0, 100.0), n_bins_grid=(3, 5)
)

# Predições do mesmo modelo C1 selecionado na grade interna (n_bins=3, ess=100, 7 parâmetros)
from noshow.experiments import fit_predict
p_c1_sel, mb_c1_sel, params_c1_sel = fit_predict(
    Xtr_c, ytr_c, Xte_c, cat_temp, cont_temp, scoring='bic-d',
    ess=float(res_c1_grid['ess']), n_bins=int(res_c1_grid['n_bins'])
)
manto_c1_str = ', '.join(mb_c1_sel)

mask_no_id = te_meta['patient_id'].isna().to_numpy()
mask_has_id = te_meta['patient_id'].notna().to_numpy()
mask_1_9 = (te_meta['patient_id'].notna() & te_meta['n_agendamentos_anteriores'].between(1, 9)).to_numpy()

ci_has_id_roc, ci_has_id_ap = bootstrap_ci(yte_c[mask_has_id], p_c1_sel[mask_has_id])
ci_1_9_roc, ci_1_9_ap = bootstrap_ci(yte_c[mask_1_9], p_c1_sel[mask_1_9])

rows_part1 = [
    {
        'Cenário / Modelo': 'Clínica Temporal (C3 SMOTENC s/ esc., BIC — protocolo completo)',
        'n': len(yte_c),
        'Prevalência': float(yte_c.mean()),
        'roc_auc': float(res_c3_sem['roc_auc']),
        'ap': float(res_c3_sem['pr_auc']),
        'lift_ap': float(res_c3_sem['pr_auc'] / yte_c.mean()),
        'brier': float(res_c3_sem['brier']),
        'prec_top5': np.nan,
        'rec_top5': np.nan,
        'prec_top10': float(res_c3_sem['precision@top10']),
        'rec_top10': float(res_c3_sem['recall@top10']),
        'Manto': res_c3_sem['manto'],
        'Params': float(res_c3_sem['n_params']),
    },
    evaluate_predictions('Clínica Temporal (C1 s/ SMOTENC, BIC — seleção interna 3 faixas)', yte_c, p_c1_sel, manto_c1_str, params_c1_sel),
    evaluate_predictions('Clínica Subgrupo (C1 3 faixas): SEM patient_id (apagão cadastral)', yte_c[mask_no_id], p_c1_sel[mask_no_id], manto_c1_str, params_c1_sel),
    evaluate_predictions('Clínica Subgrupo (C1 3 faixas): Todos COM patient_id', yte_c[mask_has_id], p_c1_sel[mask_has_id], manto_c1_str, params_c1_sel),
    evaluate_predictions('Clínica Subgrupo (C1 3 faixas): COM patient_id (1–9 cons., exploratório)', yte_c[mask_1_9], p_c1_sel[mask_1_9], manto_c1_str, params_c1_sel),
]

df_part1 = pd.DataFrame(rows_part1)
print(f'IC95% Bootstrap (C1 3 faixas) — Todos COM patient_id (n={mask_has_id.sum()}): ROC-AUC [{ci_has_id_roc[0]:.4f}, {ci_has_id_roc[1]:.4f}] | AP [{ci_has_id_ap[0]:.4f}, {ci_has_id_ap[1]:.4f}]')
print(f'IC95% Bootstrap (C1 3 faixas) — COM patient_id 1-9 cons. (n={mask_1_9.sum()}): ROC-AUC [{ci_1_9_roc[0]:.4f}, {ci_1_9_roc[1]:.4f}] | AP [{ci_1_9_ap[0]:.4f}, {ci_1_9_ap[1]:.4f}]\n')
display(df_part1.round(4))

---
## Parte 2: Validação Externa em Base Pública Real (`OpenML id=43439` / Hoppen, 2016)

Para avaliar o comportamento do classificador quando o sistema de agendamento registra o carimbo temporal da marcação (`ScheduledDay`), utilizamos o espelho público **OpenML `id=43439`** ($n = 110.526$ registros válidos após remoção de $5$ linhas com idade negativa ou antecedência negativa), derivado da base de agendamentos ambulatoriais de Vitória/ES disponibilizada originalmente por Joni Hoppen (2016) no Kaggle.

**Aspectos metodológicos importantes desta base:**
- **Ausência de `PatientId` na cópia OpenML `43439`:** Diferentemente do arquivo original do Kaggle, a versão catalogada no OpenML removeu os identificadores `PatientId` e `AppointmentID`. Consequentemente, **não há variáveis de histórico individual** nesta avaliação e o particionamento utiliza `StratifiedKFold` de 5 dobras.
- **Natureza de `SMS_received`:** A variável indica se o paciente recebeu um SMS de lembrete (e não uma resposta de confirmação). Na base, a taxa de ausência no agregado é **maior** entre quem recebeu SMS ($27{,}6\%$) do que entre quem não recebeu ($16{,}7\%$), porque o SMS não era enviado a agendamentos com até 2 dias de antecedência; dentro da mesma faixa de espera, quem recebeu SMS faltou menos.
- **Comparação com baselines (`dias_antecedencia` isolado e Regressão Logística):** Para separar o ganho proporcionado pela presença da variável `dias_antecedencia` na base do efeito do classificador, avaliamos nas mesmas 5 dobras:
  1. O preditor `dias_antecedencia` isolado (univariado);
  2. Uma **Regressão Logística** padrão com codificação *one-hot* nas mesmas variáveis discretizadas;
  3. A **Rede Bayesiana com Manto de Markov** sob `BIC` e `AIC`.

In [ ]:
ext_path = ROOT / 'data/external/vitoria_es_openml43439.csv'
ext_path.parent.mkdir(parents=True, exist_ok=True)
if ext_path.exists():
    vit = pd.read_csv(ext_path)
else:
    vit = fetch_openml(data_id=43439, as_frame=True, parser='auto').frame
    vit.to_csv(ext_path, index=False)

vit = vit[vit['Age'] >= 0].copy().reset_index(drop=True)
sched = pd.to_datetime(vit['ScheduledDay']).dt.tz_localize(None).dt.normalize()
appt = pd.to_datetime(vit['AppointmentDay']).dt.tz_localize(None).dt.normalize()
vit['dias_antecedencia'] = (appt - sched).dt.days.clip(lower=0)
vit['mesmo_dia'] = (vit['dias_antecedencia'] == 0).astype(str)
vit['dia_semana'] = appt.dt.day_name().astype(str)
vit['faixa_etaria'] = pd.cut(
    vit['Age'], [-1, 5, 17, 39, 59, 120], labels=['0-5', '6-17', '18-39', '40-59', '60+']
).astype(str)
for c in ['Gender', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received']:
    vit[c] = vit[c].astype(str)

y_vit = (vit['No-show'].astype(str).str.lower() == 'yes').astype(int).to_numpy()
cat_vit = ['Gender', 'faixa_etaria', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'dia_semana', 'mesmo_dia']
cont_vit = ['Age', 'dias_antecedencia']
X_vit = vit[cat_vit + cont_vit].reset_index(drop=True)

sms_rate = y_vit[vit['SMS_received'] == '1'].mean()
no_sms_rate = y_vit[vit['SMS_received'] == '0'].mean()
print(f'OpenML 43439 (n={len(vit):,}): Prevalência={y_vit.mean():.2%} | Falta c/ SMS={sms_rate:.1%} vs s/ SMS={no_sms_rate:.1%}')

from noshow.experiments import fit_predict, prepare_discrete
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows_part2 = []

# 1) Preditor isolado dias_antecedencia (ranking direto)
lead_score = vit['dias_antecedencia'].to_numpy(dtype=float)
lead_prob = (lead_score - lead_score.min()) / (lead_score.max() - lead_score.min() + 1e-9)
r_lead = evaluate_predictions('OpenML 43439: dias_antecedencia isolado (univariado)', y_vit, lead_prob, 'dias_antecedencia', 1.0)
r_lead['brier'] = np.nan
rows_part2.append(r_lead)

# 2) Regressão Logística nas mesmas variáveis discretizadas por fold
lr_folds = []
for tr, te in skf.split(X_vit, y_vit):
    A_d, B_d, _ = prepare_discrete(X_vit.iloc[tr], X_vit.iloc[te], cat_vit, cont_vit, n_bins=5)
    ohe = OneHotEncoder(drop='first', handle_unknown='ignore')
    Xtr_o = ohe.fit_transform(A_d)
    Xte_o = ohe.transform(B_d)
    lr = LogisticRegression(max_iter=500, random_state=42).fit(Xtr_o, y_vit[tr])
    p_lr = lr.predict_proba(Xte_o)[:, 1]
    lr_folds.append({'n_params': Xtr_o.shape[1] + 1, **evaluate_predictions('LR', y_vit[te], p_lr, '', 0)})
df_lr = pd.DataFrame(lr_folds)
rows_part2.append({
    'Cenário / Modelo': 'OpenML 43439: Regressão Logística (5-fold CV)',
    'n': len(vit),
    'Prevalência': float(y_vit.mean()),
    **{k: float(df_lr[k].mean()) for k in ['roc_auc', 'ap', 'lift_ap', 'brier', 'prec_top5', 'rec_top5', 'prec_top10', 'rec_top10']},
    'Manto': '12 variáveis (linear)',
    'Params': float(df_lr['n_params'].mean()),
})

# 3) Rede Bayesiana + Manto de Markov (BIC e AIC)
for crit_label, scoring in [('BIC', 'bic-d'), ('AIC', 'aic-d')]:
    fold_metrics = []
    for f, (tr, te) in enumerate(skf.split(X_vit, y_vit)):
        p, mb, n_params = fit_predict(X_vit.iloc[tr], y_vit[tr], X_vit.iloc[te], cat_vit, cont_vit, scoring=scoring)
        fold_metrics.append({'manto': ', '.join(mb), 'n_params': n_params, **evaluate_predictions('', y_vit[te], p, '', n_params)})
    df_f = pd.DataFrame(fold_metrics)
    row = {
        'Cenário / Modelo': f'OpenML 43439: Rede Bayesiana + Manto (5-fold CV, {crit_label})',
        'n': len(vit),
        'Prevalência': float(y_vit.mean()),
        **{k: float(df_f[k].mean()) for k in ['roc_auc', 'ap', 'lift_ap', 'brier', 'prec_top5', 'rec_top5', 'prec_top10', 'rec_top10']},
        'Manto': df_f['manto'].mode().iloc[0],
        'Params': float(df_f['n_params'].mean()),
    }
    rows_part2.append(row)
    print(f'Vitória/ES ({crit_label}): ROC-AUC={df_f["roc_auc"].mean():.4f}±{df_f["roc_auc"].std():.4f} | AP={df_f["ap"].mean():.4f}±{df_f["ap"].std():.4f} (Lift={df_f["lift_ap"].mean():.2f}x) | P@5%={df_f["prec_top5"].mean()*100:.1f}% | P@10%={df_f["prec_top10"].mean()*100:.1f}% | Params={df_f["n_params"].mean():.0f} | Manto={row["Manto"]}')

df_part2 = pd.DataFrame(rows_part2)
display(df_part2.round(4))

---
## Parte 3: Teste de Recuperação Estrutural sob Processo Gerador Conhecido (Cenário 1B: $X \to Y_{\text{sim}}$ contra Oráculo Teórico)

Para verificar se o algoritmo de aprendizado estrutural (*Hill-Climb Search* + Manto de Markov) recupera preditores relevantes e descarta variáveis periféricas sem qualquer vazamento de alvo:
- **Por que não usamos condicionamento reverso ($Y \to X$):** Sortear preditores condicionados ao desfecho real observado criaria dependência circular (*target leakage* por construção).
- **Desenho do Cenário 1B ($X \to Y_{\text{sim}}$) como teste de recuperação estrutural:**
  1. A estrutura de agendamentos, datas, clima e especialidades da clínica ($n = 49.593$) é mantida.
  2. Os coeficientes do gerador são **parâmetros sintéticos arbitrários escolhidos para definir um processo gerador conhecido** (e não calibrados por literatura externa). A probabilidade verdadeira $p_{\text{true}, i} = \sigma(\eta_i)$ é gerada no sentido direto ($X \to Y_{\text{sim}}$) a partir de **três fatores principais** (risco latente longitudinal refletido no histórico `taxa_faltas_anteriores`, antecedência de marcação `dias_antecedencia` e status de confirmação `confirmacao_previa`), mais um efeito secundário de chuva intensa (`max_rain_day > 15`).
  3. O modelo recebe todas as **26 variáveis candidatas na avaliação temporal (27 na validação cruzada)** e é comparado contra o **Oráculo Teórico do Gerador** ($p_{\text{true}}$) tanto em **AP** (métrica principal) quanto em **ROC-AUC**.

In [ ]:
from noshow.evaluation import patient_groups, repeated_group_folds
from noshow.experiments import fit_predict

df_base = df_clinic.copy()

def build_semisynthetic_1b_forward(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """Simulação causal direta X -> Y_sim (sem condicionamento reverso) e registro de prob_true do oráculo."""
    rng = np.random.default_rng(seed)
    sim = df.copy()
    missing_mask = sim['patient_id'].isna().to_numpy()
    n_missing = int(missing_mask.sum())
    pat_pool = np.arange(int(sim['patient_id'].max()) + 1000, int(sim['patient_id'].max()) + 1000 + max(1, n_missing // 12))
    sim.loc[missing_mask, 'patient_id'] = rng.choice(pat_pool, size=n_missing)
    sim['patient_id'] = sim['patient_id'].astype('Int64')
    sim['dias_antecedencia'] = np.round(np.clip(rng.gamma(shape=1.8, scale=8.5, size=len(sim)), 0, 120), 0)

    unique_pids = sim['patient_id'].unique()
    pat_risk = pd.Series(rng.normal(0, 1.15, size=len(unique_pids)), index=unique_pids)
    pat_eta = sim['patient_id'].map(pat_risk).to_numpy()
    score_conf = -0.6 * pat_eta - 0.025 * (sim['dias_antecedencia'].to_numpy() - 15) + rng.normal(0, 0.8, len(sim))
    sim['confirmacao_previa'] = np.where(
        score_conf > -0.35, 'confirmou', np.where(score_conf > -1.25, 'sem_resposta', 'duvida')
    )
    conf_effect = np.where(
        sim['confirmacao_previa'] == 'confirmou', -0.95, np.where(sim['confirmacao_previa'] == 'sem_resposta', 0.65, 1.15)
    )
    rain_effect = 0.25 * (sim['max_rain_day'].fillna(0).to_numpy() > 15.0)
    logit = -2.85 + 0.90 * pat_eta + 0.032 * sim['dias_antecedencia'].to_numpy() + conf_effect + rain_effect
    prob_true = 1.0 / (1.0 + np.exp(-logit))
    sim['prob_true'] = prob_true
    sim['no_show'] = (rng.random(len(sim)) < prob_true).astype(int)
    sim = add_patient_history(sim)
    return sim

sim_1b = build_semisynthetic_1b_forward(df_base)
cat_cols = CATEGORICAL + ['confirmacao_previa']
cont_cols = CONTINUOUS + ['dias_antecedencia']
y_1b = sim_1b['no_show'].to_numpy()
X_1b = sim_1b[cat_cols + cont_cols]

rows_part3 = []
# Oráculo teórico do gerador (prob_true exata)
ano_1b = sim_1b['appointment_date'].dt.year
tr_1b, te_1b = np.where(ano_1b <= 2020)[0], np.where(ano_1b >= 2021)[0]
row_oracle_cv = evaluate_predictions(
    '1B Oráculo Teórico do Gerador (p_true exata — Base Completa)',
    y_1b, sim_1b['prob_true'].to_numpy(), 'gerador exato (limite teórico)', 0.0
)
row_oracle_te = evaluate_predictions(
    '1B Oráculo Teórico do Gerador (p_true exata — Temporal 2021–22)',
    y_1b[te_1b], sim_1b['prob_true'].to_numpy()[te_1b], 'gerador exato (limite teórico)', 0.0
)
rows_part3.append(row_oracle_te)

# CV Agrupada 5-fold (BIC)
cv_rows = []
for r, f, tr, te in repeated_group_folds(y_1b, patient_groups(sim_1b['patient_id']), n_splits=5, n_repeats=1):
    p, mb, n_params = fit_predict(X_1b.iloc[tr], y_1b[tr], X_1b.iloc[te], cat_cols, cont_cols, scoring='bic-d')
    cv_rows.append({'manto': ', '.join(mb), 'n_params': n_params, **evaluate_predictions('', y_1b[te], p, '', n_params)})
df_cv_1b = pd.DataFrame(cv_rows)
rows_part3.append({
    'Cenário / Modelo': '1B Causal Direto (X->Y_sim) — CV Agrupada (BIC)',
    'n': len(sim_1b),
    'Prevalência': float(y_1b.mean()),
    **{k: float(df_cv_1b[k].mean()) for k in ['roc_auc', 'ap', 'lift_ap', 'brier', 'prec_top5', 'rec_top5', 'prec_top10', 'rec_top10']},
    'Manto': df_cv_1b['manto'].mode().iloc[0],
    'Params': float(df_cv_1b['n_params'].mean()),
})

# Temporal 2021-2022 (BIC e AIC)
cat_t_1b = [c for c in cat_cols if c not in {'appointment_year', 'periodo'}]
for crit, sc in [('BIC', 'bic-d'), ('AIC', 'aic-d')]:
    p_t, mb_t, np_t = fit_predict(X_1b.iloc[tr_1b], y_1b[tr_1b], X_1b.iloc[te_1b], cat_t_1b, cont_cols, scoring=sc)
    r_t = evaluate_predictions(
        f'1B Causal Direto (X->Y_sim) — Temporal 2021–22 ({crit})',
        y_1b[te_1b], p_t, ', '.join(mb_t), float(np_t)
    )
    rows_part3.append(r_t)
    pct_roc = (r_t['roc_auc'] / row_oracle_te['roc_auc']) * 100
    pct_ap = (r_t['ap'] / row_oracle_te['ap']) * 100
    print(f'Candidatas CV={len(cat_cols)+len(cont_cols)}, Temporal={len(cat_t_1b)+len(cont_cols)} | 1B Temporal ({crit}): ROC-AUC={r_t["roc_auc"]:.4f} ({pct_roc:.1f}% do Oráculo {row_oracle_te["roc_auc"]:.4f}) | AP={r_t["ap"]:.4f} ({pct_ap:.1f}% do Oráculo {row_oracle_te["ap"]:.4f}) | Manto={r_t["Manto"]} ({int(r_t["Params"])} params)')

df_part3 = pd.DataFrame(rows_part3)
display(df_part3.round(4))

In [ ]:
# Tabela consolidada final e exportação para CSV
df_all = pd.concat([df_part1, df_part2, df_part3], ignore_index=True)
out_csv = ROOT / 'resultados/validacao_sensibilidade_e_base_externa.csv'
df_all.to_csv(out_csv, index=False)
print('Tabela consolidada salva em:', out_csv)
display(df_all.round(4))

# Visualização comparativa: ROC-AUC e Lift de AP (AP / Prevalência)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

plot_rows = [
    ('Clínica: Sem ID (acaso)', df_part1.iloc[2]['roc_auc'], df_part1.iloc[2]['lift_ap'], '#9e9e9e'),
    ('Clínica: Temp. C3 (BIC)', df_part1.iloc[0]['roc_auc'], df_part1.iloc[0]['lift_ap'], '#64b5f6'),
    ('Clínica: Com ID (n=5.846)', df_part1.iloc[3]['roc_auc'], df_part1.iloc[3]['lift_ap'], '#1e88e5'),
    ('OpenML 43439: Lead Time só', df_part2.iloc[0]['roc_auc'], df_part2.iloc[0]['lift_ap'], '#ffb74d'),
    ('OpenML 43439: Reg. Logística', df_part2.iloc[1]['roc_auc'], df_part2.iloc[1]['lift_ap'], '#fb8c00'),
    ('OpenML 43439: Rede Bayes (AIC)', df_part2.iloc[3]['roc_auc'], df_part2.iloc[3]['lift_ap'], '#e65100'),
    ('1B Gerador: Rede Bayes (AIC)', df_part3.iloc[3]['roc_auc'], df_part3.iloc[3]['lift_ap'], '#43a047'),
    ('1B Gerador: Oráculo Teórico', df_part3.iloc[0]['roc_auc'], df_part3.iloc[0]['lift_ap'], '#1b5e20'),
]

labels = [r[0] for r in plot_rows]
rocs = [r[1] for r in plot_rows]
lifts = [r[2] for r in plot_rows]
colors = [r[3] for r in plot_rows]
y_pos = np.arange(len(labels))

axes[0].barh(y_pos, rocs, color=colors, edgecolor='black', height=0.65)
axes[0].axvline(0.50, color='red', linestyle='--', linewidth=1, label='Acaso (0,50)')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(labels)
axes[0].set_xlim(0.45, 0.92)
axes[0].set_xlabel('ROC-AUC')
axes[0].set_title('Discriminação (ROC-AUC) por Cenário e Baseline')
for i, v in enumerate(rocs):
    axes[0].text(v + 0.008, i, f'{v:.4f}', va='center', fontsize=8.5, fontweight='bold')

axes[1].barh(y_pos, lifts, color=colors, edgecolor='black', height=0.65)
axes[1].axvline(1.00, color='red', linestyle='--', linewidth=1, label='Prevalência (1,00x)')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([''] * len(labels))
axes[1].set_xlim(0.85, 3.45)
axes[1].set_xlabel('Lift de AP (AP / Prevalência)')
axes[1].set_title('Ganho de Precisão Média Normalizado pela Prevalência')
for i, v in enumerate(lifts):
    axes[1].text(v + 0.04, i, f'{v:.2f}x', va='center', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Síntese Metodológica dos Resultados

1. **Efeito da Lacuna Cadastral na Base Primária (2021–2022, mesmo modelo `C1` da Tabela 1: 3 faixas, `ESS = 100`, 7 parâmetros):**
   - Nos $4.104$ registros sem `patient_id` ($41{,}2\%$ do teste temporal), todas as variáveis históricas são ausentes e o modelo emite probabilidade constante (`ROC-AUC = 0,5000`, `Lift de AP = 1,00x`, `P@10% = 16,5%`).
   - Nos $5.846$ registros com `patient_id` válido, a `ROC-AUC` sobe para **$0{,}5712$** (IC95% `[0,552; 0,591]`, `AP = 0,1694`, `Lift de AP = 1,17x`, `P@10% = 17,1%`). No subgrupo exploratório de 1 a 9 consultas ($n = 676$), obtém-se `ROC-AUC = 0,5949` (IC95% `[0,546; 0,646]`, `AP = 0,2244`, `Lift de AP = 1,21x`).
2. **Validação Externa na Base Pública `OpenML id=43439` (Hoppen, 2016, $n = 110.526$):**
   - A presença da variável `dias_antecedencia` (*lead time*) é o principal determinante do salto de `ROC-AUC` para $\sim 0{,}72$: sozinha, `dias_antecedencia` já alcança `ROC-AUC = 0,6982` (`AP = 0,3050`, `Lift de AP = 1,51x`).
   - A Rede Bayesiana com Manto de Markov sob AIC (`ROC-AUC = 0,7230 ± 0,0021`, `AP = 0,3459`, `Lift de AP = 1,71x`, `P@5% = 42,1%`, `P@10% = 39,5%`, `215` parâmetros) e sob BIC (`ROC-AUC = 0,7192 ± 0,0022`, `AP = 0,3356`, `Lift de AP = 1,66x`, `71` parâmetros) fica próxima da Regressão Logística com 12 variáveis (`ROC-AUC = 0,7256`, `AP = 0,3543`, `Lift de AP = 1,75x`).
   - Quando normalizada pela prevalência ($20{,}19\%$ contra $9{,}74\%$ na clínica), a maior `AP` nominal reflete em grande parte a taxa basal da classe positiva (`Lift de AP = 1,71x` em Vitória contra `1,46x` do `C1` e `1,81x` do `C0+S` na validação cruzada da clínica).
3. **Recuperação Estrutural sob Processo Gerador Conhecido (Cenário 1B: $X \to Y_{\text{sim}}$, coeficientes sintéticos arbitrários):**
   - Contra o Oráculo Teórico do gerador no período temporal (`ROC-AUC = 0,8733`, `AP = 0,5788`, `Lift de AP = 3,82x`), a Rede Bayesiana avaliada sobre as **26 candidatas temporais (27 na validação cruzada)** recupera:
     - **Em AP (métrica principal):** **`81,4%` do oráculo** no BIC (`0,4713` contra `0,5788`) e **`86,1%` do oráculo** no AIC (`0,4981` contra `0,5788`);
     - **Em ROC-AUC:** **`97,2%` do oráculo** no BIC (`0,8489`, retendo 2 variáveis, `59` parâmetros) e **`97,8%` do oráculo** no AIC (`0,8537`, isolando as **3 variáveis de maior efeito** (`confirmacao_previa, dias_antecedencia, taxa_faltas_anteriores`, `334` parâmetros) e descartando as demais variáveis periféricas e o termo secundário de chuva).